In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

In [1]:
# This cell is hidden from users
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
instance = service.active_account()["instance"]
backend_name = service.least_busy(operational=True, min_num_qubits=16).name

> **Note:** Funcțiile Qiskit sunt o funcționalitate experimentală disponibilă numai utilizatorilor IBM Quantum&reg; Premium Plan, Flex Plan și On-Prem (prin IBM Quantum Platform API) Plan. Acestea se află în stadiu de previzualizare și pot suferi modificări.

## Prezentare generală
În chimia cuantică, problema structurii electronice se concentrează pe găsirea soluțiilor ecuației Schrödinger electronice — funcțiile de undă cuantice care descriu comportamentul electronilor sistemului. Aceste funcții de undă sunt vectori de amplitudini complexe, fiecare amplitudine corespunzând contribuției unei configurații electronice posibile.

Starea fundamentală este funcția de undă cu cea mai mică energie a sistemului și are o importanță deosebită în studiul sistemelor moleculare. Abordarea cea mai precisă pentru calcularea stării fundamentale ia în considerare toate configurațiile electronice posibile, însă aceasta devine imposibil de gestionat pentru sisteme mai mari, deoarece numărul configurațiilor crește exponențial cu dimensiunea sistemului.

Handover Iterative Variational Quantum Eigensolver (HI-VQE) este o metodă hibridă cuantic-clasică inovatoare pentru estimarea precisă a stării fundamentale a sistemelor moleculare. Integrează hardware cuantic cu calcul clasic, utilizând procesoare cuantice pentru a explora eficient configurațiile electronice candidate și calculând funcția de undă rezultantă pe calculatoare clasice. Prin generarea de funcții de undă compacte, dar precise din punct de vedere chimic, HI-VQE îmbunătățește cercetarea și descoperirile în chimia cuantică și știința materialelor.

![Imagine care prezintă o privire de ansamblu asupra algoritmului HI-VQE al Qunova](../docs/images/guides/qunova-chemistry/overview.svg)

HI-VQE reduce complexitatea computațională a problemei structurii electronice prin estimarea eficientă a stării fundamentale cu o precizie ridicată. Se concentrează pe un subset atent selectat al celor mai relevante configurații electronice, optimizând atât precizia, cât și eficiența.

Combinând punctele forte ale calculatoarelor clasice și cuantice, HI-VQE rafinează și îmbunătățește iterativ estimarea curentă a funcției de undă. Tehnicile sale unice de construire a subspatiului ajută la eficientizarea selecției configurațiilor, astfel încât utilizatorii să aibă un control computational mai mare și o precizie îmbunătățită în simulările de chimie cuantică.

Dacă dorești să înveți mai în profunzime despre algoritm, poți [citi lucrarea de cercetare asociată](https://arxiv.org/abs/2503.06292).

## Descriere
Numărul de configurații electronice pentru un sistem molecular crește exponențial cu dimensiunea sistemului. Cu toate acestea, pentru anumite stări electronice, cum ar fi starea fundamentală, este obișnuit ca doar o mică fracțiune din configurații să contribuie semnificativ la energia stării. Metodele de interacțiune de configurație selectată (SCI) exploatează această sparsitate pentru a reduce costurile computaționale prin identificarea și concentrarea pe cele mai relevante configurații. Acest subset de configurații este denumit subspațiu.

HI-VQE valorifică eficiența inerentă a calculatoarelor cuantice pentru reprezentarea sistemelor moleculare în scopul facilitării căutării în subspațiu. Integrează subrutine clasice și cuantice pentru a rezolva problema structurii electronice cu o precizie ridicată. Spre deosebire de metodele SCI cuantice existente, HI-VQE combină antrenamentul variațional, construirea iterativă a subspațiului și filtrarea configurațiilor prin pre-diagonalizare pentru a îmbunătăți eficiența prin reducerea măsurătorilor cuantice, a iterațiilor și a costurilor de diagonalizare clasică. HI-VQE poate fi astfel aplicat unor sisteme moleculare mai mari care necesită mai mulți Qubiți și reduce costul rezolvării unei probleme de o dimensiune dată la același grad de precizie.

![Imagine care prezintă o descriere detaliată a fiecărui pas din algoritmul HI-VQE al Qunova.](../docs/images/guides/qunova-chemistry/description.avif)

Pentru a calcula starea fundamentală a unui sistem, HI-VQE utilizează mai întâi pachetul clasic de chimie PySCF pentru a genera o reprezentare moleculară din datele de intrare furnizate de utilizator, cum ar fi geometria moleculară și alte informații moleculare. Intră apoi într-o buclă de optimizare hibridă cuantic-clasică, rafinând iterativ un subspațiu pentru a reprezenta optim starea fundamentală, minimizând în același timp numărul de configurații incluse. Bucla continuă până când sunt îndeplinite criterii de convergență, cum ar fi dimensiunea subspațiului sau stabilitatea energiei, după care sunt furnizate funcția de undă a stării fundamentale calculate și energia. Aceste rezultate pot fi utilizate pentru a construi suprafețe de energie potențială precise și pentru a efectua analize suplimentare ale sistemului.

Bucla de optimizare se concentrează pe ajustarea parametrilor unui Circuit cuantic pentru a genera un subspațiu de înaltă calitate. HI-VQE oferă trei opțiuni de Circuit cuantic: [`excitation_preserving`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.excitation_preserving), [efficient_su2](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.efficient_su2) și [LUCJ](https://qiskit-community.github.io/ffsim/explanations/lucj.html). Optimizarea este inițializată aproape de starea de referință Hartree-Fock datorită adecvabilității sale generale. Circuit-ul este apoi executat pe un dispozitiv cuantic și configurațiile sunt eșantionate din starea cuantică rezultantă înainte de a fi returnate ca șiruri binare. Datorită zgomotului dispozitivului cuantic, unele configurații eșantionate pot fi invalide din punct de vedere fizic, neconservând numărul de electroni sau spinul. HI-VQE abordează aceasta folosind procesul de recuperare a configurațiilor din pachetul [qiskit-addon-sqd](/guides/qiskit-addons-sqd#sample-based-quantum-diagonalization-sqd-overview), astfel încât utilizatorii pot fie să corecteze configurațiile invalide, fie să le elimine.

Configurațiile valide parcurg apoi un pas opțional de filtrare pentru a le elimina pe cele estimate să contribuie minim. Aceasta reduce dimensiunea subspațiului, scăzând astfel costul pasului de diagonalizare. Dacă filtrarea este activată, atunci se construiește un Hamiltonian de subspațiu preliminar din configurațiile valide și se efectuează o diagonalizare cu criterii de terminare foarte relaxate. Deși precizia amplitudinilor rezultante pentru fiecare configurație este scăzută, aceasta este eficientă pentru a prezice ce configurații să fie excluse din subspațiu în această iterație și este rapidă de calculat.

Configurațiile selectate sunt adăugate la subspațiu, iar Hamiltonianul sistemului este proiectat în acest subspațiu. Subspațiul se actualizează iterativ, păstrând cele mai relevante configurații de-a lungul iterațiilor. Această abordare contrastează cu metodele alternative deoarece Circuit-ul cuantic nu trebuie să aproximeze întreaga stare fundamentală la fiecare pas.

Ulterior, Hamiltonianul de subspațiu este diagonalizat clasic pentru a obține cea mai mică valoare proprie și vectorul propriu corespunzător, reprezentând o aproximare a stării fundamentale și a energiei sale. Pe măsură ce calitatea subspațiului se îmbunătățește prin iterații, starea fundamentală calculată aproximează mai bine starea fundamentală reală. Un pas suplimentar de filtrare poate fi efectuat în acest punct pentru a elimina din subspațiu orice configurații care nu contribuie substanțial la starea fundamentală calculată. Acest pas asigură că subspațiul transferat în iterația următoare este cât mai compact posibil. Aceasta este evaluată pe baza amplitudinilor returnate de diagonalizare, deoarece acestea reprezintă contribuția de importanță a fiecărei configurații la starea fundamentală calculată.

O verificare a convergenței determină apoi dacă antrenamentul suplimentar ar îmbunătăți rezultatele. Dacă da, se efectuează un pas opțional de expansiune clasică, parametrii Circuit-ului cuantic sunt actualizați pentru a minimiza în continuare energia calculată și procesul se repetă. Pasul de expansiune clasică generează configurații suplimentare pentru subspațiu, completând configurațiile eșantionate de pe dispozitivul cuantic. Identifică mai întâi configurația cu amplitudinea cea mai mare în rezultatele diagonalizării, înainte de a genera noi configurații cu excitații simple și duble din configurația identificată. Numărul dorit din aceste configurații este apoi adăugat la subspațiu.

Odată ce s-a determinat că iterațiile au convergat, HI-VQE returnează starea fundamentală calculată (sub forma stărilor din subspațiu și a amplitudinilor lor în funcția de undă a stării fundamentale), energia sa și o măsură a varianței energiei care indică dacă starea calculată formează o stare proprie a Hamiltonianului sistemului.

Utilizatorii pot decide Circuit-ul cuantic utilizat și numărul de shot-uri efectuate pentru fiecare Circuit cuantic, precum și controla dimensiunea subspațiului sau activa generarea clasică de configurații suplimentare pentru a asista configurațiile generate cuantic. În acest fel, utilizatorii pot adapta comportamentul HI-VQE pentru aplicațiile dorite.

## Noțiuni introductive
Mai întâi, [solicită acces la funcție](https://forms.office.com/r/zN3hvMTqJ1).
Apoi, autentifică-te folosind [cheia ta API IBM Quantum&reg;](http://quantum.cloud.ibm.com/) și, presupunând că ți-ai [salvat deja contul](/guides/functions#install-qiskit-functions-catalog-client) în mediul local, selectează Funcția Qiskit după cum urmează:

In [ ]:
import reprlib
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

function = catalog.load("qunova/hivqe-chemistry")

## Inputs

Consultă tabelul următor pentru toți parametrii de intrare acceptați de funcție. Secțiunile ulterioare [Opțiuni moleculă](#molecule-options) și [Opțiuni HI-VQE](#hi-vqe-options) oferă mai multe detalii despre aceste argumente.
| Nume                   | Tip                                                           | Descriere                                                                                                                                                                                                                                                                                                 | Obligatoriu | Implicit                                                                  | Exemplu                                                                                   |
|------------------------|----------------------------------------------------------------|-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|----------|--------------------------------------------------------------------------|-------------------------------------------------------------------------------------------|
| geometry               | Union[List[List[Union[str, Tuple[float, float, float]]]], str] | Poate fi fie un șir de caractere, fie liste structurate care conțin perechi de atomi și coordonate. Dacă este furnizat ca șir de caractere, trebuie să fie o geometrie moleculară în Format de Coordonate Carteziene. Dacă este furnizat ca listă, trebuie să fie o listă de liste, fiecare conținând un șir pentru atom și un tuplu de coordonate. | Da      | N/A                                                                      | `[['O', (0, 0, 0)], ['H', (0, 1, 0)], ['H', (0, 0, 1)]]` sau `"O 0 0 0; H 0 1 0; H 0 0 1"` |
| backend\_name          | str                                                            | Numele Backend-ului pentru care se face interogarea.                                                                                                                                                                                                                                                      | Da      | N/A                                                                      | `ibm_fez`                                                                                 |
| max\_states            | int                                                            | Dimensiunea maximă a subspațiului pentru diagonalizare. Se vor folosi mai puține stări dacă numărul nu este un pătrat perfect.                                                                                                                                                                                                                                                    | Da      | N/A                                                                      | `100`                                                                                     |
| max\_expansion\_states | int                                                            | Numărul maxim de stări CI generate clasic care vor fi incluse în fiecare iterație.                                                                                                                                                                                                                     | Da      | N/A                                                                      | `10`                                                                                      |
| molecule_options                | dict                                                           | Opțiuni legate de molecula folosită ca intrare pentru HI-VQE. Consultă secțiunea [Opțiuni moleculă](#molecule-options) pentru mai multe detalii.                                                                                                                                                                                                                                                | Nu       | Consultă secțiunea [Opțiuni moleculă](#molecule-options) pentru detalii.                                 | `{"basis": "sto3g", "unit": "angstrom" }`                               |
| hivqe_options                | dict                                                           | Opțiuni care controlează comportamentul algoritmului HI-VQE. Consultă secțiunea [Opțiuni HI-VQE](#hi-vqe-options) pentru mai multe detalii.                                                                                                                                                                                                                                                | Nu       | Consultă secțiunea [Opțiuni HI-VQE](#hi-vqe-options) pentru detalii.                                 | `{"shots": 10_000, "max_iter": 10 }`                               |
### Opțiuni moleculă
Tabelul următor detaliază toate cheile și valorile care pot fi setate în dicționarul `molecule_options`, precum și tipurile de date și valorile implicite ale acestora. Toate cheile sunt opționale.

| Cheie                             | Tip valoare                         | Valoare implicită                | Interval valid                                                                                                                                                 | Explicație                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                             |
|:----------------------------------|:-----------------------------------:|:--------------------------------:|:---------------------------------------------------------------------------------------------------------------------------------------------------------------|:----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| `"charge"`                        | `int`                               | `0`                              | Diverse                                                                                                                                                        | Un număr întreg care specifică sarcina netă totală a sistemului molecular. Valoarea implicită este 0; totuși, poate fi orice număr întreg.                                                                                                                                                                                                                                                                                                                                                                                              |
| `"basis"`                         | `str`                               | `'sto-3g'`                       | Diverse                                                                                                                                                        | Un șir de caractere care specifică tipul bazei; acestea sunt transmise către pyscf. (ex: `"sto-3g"`, `"3-21g"`, `"6-31g"`, `"cc-pvdz"`)                                                                                                                                                                                                                                                                                                                                                                                                      |
| `"active_orbitals"`               | `List[int]`                         | Toți indicii orbitalilor.             | Indicii orbitalilor spațiali valizi pentru problemă.                                                                                                             | O listă de indici ai orbitalilor activi în intervalul [0, n), unde n este numărul de Qubiți folosiți în problemă. Dacă este specificat, atunci argumentul frozen_orbitals trebuie să fie și el specificat.                                                                                                                                                                                                                                                                                                                            |
| `"frozen_orbitals"`               | `List[int]`                         | Niciun indice.                      | Indicii orbitalilor spațiali valizi pentru problemă, excluzând orbitalii activi.                                                                                  | O listă de indici ai orbitalilor înghețați în același interval ca active_orbitals. Dacă este specificat, atunci active_orbitals trebuie să fie și el specificat. Reține că doar orbitalii ocupați ar trebui înghețați, deoarece numărul de electroni activi scade cu 2 pentru fiecare orbital ocupat care este înghețat.                                                                                                                                                                                                                        |
| `"orbital_coeffs"`                | `List[List[float]]`                 | Orbitalii moleculari Hartree-Fock. | Diverse.                                                                                                                                                       | Coeficienții pentru orbitalii spațiali folosiți în calculul integralelor de repulsie electronică ale sistemului. Câteva exemple valide sunt orbitalii moleculari Hartree-Fock, orbitalii naturali și orbitalii AVAS.                                                                                                                                                                                                                                                                                                                   |
| `"symmetry"`                      | `Union[str, bool]`                  | `False`                          | `True` sau `False`                                                                                                                                              | Folosit pentru a invoca simetria grupului de puncte pentru calculele moleculare inițiale, în scopul construirii bazei orbitale adaptate simetriei. Acești orbitali adaptați simetriei sunt folosiți ca funcții de bază pentru calculele SCF ulterioare. Valoarea implicită este False; dacă este setat la True, atunci va fi invocat și grupurile de puncte arbitrare vor fi detectate și utilizate automat. Dacă este atribuită o simetrie particulară, de exemplu symmetry = "Dooh", atunci va fi generată o eroare dacă geometria moleculară nu respectă simetria necesară. |
| `"symmetry_subgroup"`             | `Optional[str]`                     | `None`                           | Consultă [documentația pyscf](https://pyscf.org/pyscf_api_docs/pyscf.gto.html#pyscf.gto.mole.MoleBase.build).                                                      | Poate fi folosit pentru a genera un subgrup al simetriei detectate. Nu are niciun efect când simetria este specificată folosind argumentul cuvânt cheie symmetry.                                                                                                                                                                                                                                                                                                                                                                         |
| `"unit"`                          | `str`                               | `"angstrom"`                     | Consultă [documentația pyscf](https://pyscf.org/pyscf_api_docs/pyscf.gto.html#pyscf.gto.mole.MoleBase.build).                                                      | Specifică unitatea de măsură folosită pentru coordonatele și distanțele atomice. Implicit se folosesc unități angstrom.                                                                                                                                                                                                                                                                                                                                                                                                                       |
| `"nucmod"`                        | `Optional[Union[dict, str]]`        | `None`                           | Consultă [documentația pyscf](https://pyscf.org/pyscf_api_docs/pyscf.gto.html#pyscf.gto.mole.MoleBase.build).                                                      | Specifică modelul nuclear care va fi folosit. În mod implicit, folosește modelul nuclear punctiform, iar alte valori activează modelul nuclear Gaussian. Dacă este furnizată o funcție, aceasta va fi utilizată cu modelul nuclear Gaussian pentru a genera valoarea distribuției sarcinii nucleare 'zeta'.                                                                                                                                                                                                                                                                   |
| `"pseudo"`                        | `Optional[Union[dict, str]]`        | `None`                           | Consultă [documentația pyscf](https://pyscf.org/pyscf_api_docs/pyscf.gto.html#pyscf.gto.mole.MoleBase.build).                                                      | Specifică pseudopotențialul pentru atomii din moleculă. Implicit este None, indicând că nu se aplică pseudopotențiale și toți electronii sunt incluși explicit în calcule.                                                                                                                                                                                                                                                                                                                                                                |
| `"cart"`                          | `bool`                              | `False`                          | Consultă [documentația pyscf](https://pyscf.org/pyscf_api_docs/pyscf.gto.html#pyscf.gto.mole.MoleBase.build).                                                      | Specifică dacă să se folosească GTO-uri carteziene ca funcții de bază pentru momentul unghiular în calcul. Valoarea implicită False va folosi GTO-uri sferice.                                                                                                                                                                                                                                                                                                                                                               |
| `"magmom"`                        | `Optional[List[Union[int, float]]]` | `None`                           | Consultă [documentația pyscf](https://pyscf.org/pyscf_api_docs/pyscf.gto.html#pyscf.gto.mole.MoleBase.build).                                                      | Setează momentul magnetic de spin coliniar al fiecărui atom. În mod implicit, acesta este None și fiecare atom este inițializat cu un spin zero.                                                                                                                                                                                                                                                                                                                                                                                         |
| `"avas_aolabels"`                 | `Optional[List[str]]`               | `None`                           | ex. ["H 1s", "O 2p"] pentru H2O                                                                                                                                                             | Definește Orbitalii Atomici care vor fi incluși în schema AVAS. Consultă [documentația AVAS](https://arxiv.org/abs/1701.07862).                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                         |
| `"avas_threshold"`                | `float`                             | `0.2`                            | Între 0.0 și 2.0                                                                                                                                                      | Specifică valoarea de prag folosită pentru a determina care Orbitali Atomici (AO) sunt păstrați în spațiul activ.                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                      |
| `"noons_level"`                   | `Optional[str]`                     | `None`                           | `"mp2"` sau `"ccsd"`                                                                                                                                            | Definește abordarea teoretică pentru pregătirea orbitalilor naturali și selectarea orbitalilor activi pe baza schemei Numerelor de Ocupație ale Orbitalilor Naturali (NOONs). Consultă [documentația NOONs](https://doi.org/10.1063/5.0042147). Atât indicii orbitalilor activi, cât și cei ai orbitalilor înghețați trebuie furnizați pentru a controla numărul de orbitali (și numărul de Qubiți).                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                             |
### Opțiuni HI-VQE
Tabelul următor detaliază toate cheile și valorile care pot fi setate în dicționarul `hivqe_options`, precum și tipurile de date și valorile implicite ale acestora. Toate cheile sunt opționale.

| Cheie                             | Tip valoare                         | Valoare implicită                | Interval valid                                                                                                                                                 | Explicație                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                             |
|:----------------------------------|:-----------------------------------:|:--------------------------------:|:---------------------------------------------------------------------------------------------------------------------------------------------------------------|:----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| `"shots"`                         | `int`                               | `1_000`                          | Între 1 și 10 000.                                                                                                                                          | Numărul de măsurători de folosit pe dispozitivul cuantic per iterație.                                                                                                                                                                                                                                                                                                                                                                                                                                                             |
| `"max_iter"`                      | `int`                               | `25`                             | Între 1 și 50.                                                                                                                                              | Numărul maxim de iterații efectuate pentru optimizarea ansatz-ului. Algoritmul poate folosi mai puține iterații dacă se obține convergența devreme.                                                                                                                                                                                                                                                                                                                                                                                                                 |
| `"initial_basis_states"`          | `List[str]`                         | Starea Hartree-Fock.          | Șiruri binare cu numărul de biți corespunzând numărului necesar de Qubiți pentru problemă.                                                                 | Poate fi folosit pentru a reporni algoritmul cu stări clasice dintr-un rezultat anterior.                                                                                                                                                                                                                                                                                                                                                                                                                                      |
| `"ansatz"`                        | `str`                               | `"epa"`                          | `"epa"`, `"hea"`, sau `"lucj"`                                                                                                                                  | Specifică ansatz-ul cuantic care va fi optimizat pentru a genera stări noi. `"epa"` selectează ansatz-ul cu păstrarea excitației. `"hea"` selectează ansatz-ul eficient din punct de vedere hardware. `"lucj"` selectează ansatz-ul local unitary cluster Jastrow.                                                                                                                                                                                                                                                                                                                                                                       |
| `"convergence_count"`             | `int`                               | `3`                              | Cel puțin 2.                                                                                                                                                    | Numărul de iterații fără modificare semnificativă a energiei calculate care trebuie să se scurgă înainte ca algoritmul să fie considerat convergent.                                                                                                                                                                                                                                                                                                                                                                                                                                                                         |
| `"convergence_abstol"`            | `float`                             | `1e-4`                           | Mai mare decât 0 și cel mult 1.                                                                                                                                     | Magnitudinea modificării energiei calculate care este considerată semnificativă în scopul verificărilor de convergență.                                                                                                                                                                                                                                                                                                                                                                                                                       |
| `"reset_convergence_count"`       | `bool`                              | `True`                           | `True` sau `False`                                                                                                                                              | Dacă este `True`, cele `convergence_count` iterații trebuie să aibă loc fără o modificare semnificativă care să le întrerupă pentru a fi calificate drept convergență. Dacă este `False`, atunci algoritmul se va opri după `convergence_count` dacă au apărut modificări nesemnificative la oricare iterație din procesul de optimizare.                                                                                                                                                                                                                                                 |
| `"configuration_recovery"`        | `bool`                              | `True`                           | `True` sau `False`.                                                                                                                                             | Dacă să se folosească sau nu recuperarea configurației din pachetul `qiskit-addon-sqd`. Dacă este True, stările invalide eșantionate de pe dispozitivul cuantic sunt corectate clasic. Dacă este False, acestea sunt eliminate.                                                                                                                                                                                                                                                                                                                                      |
| `"ansatz_entanglement"`           | `str`                               | `"circular"`                     | Oricare dintre `"linear"`, `"reverse_linear"`, `"pairwise"`, `"circular"`, `"full"`, sau `"sca"`. Dacă se folosește ansatz-ul `"lucj"`, `"lucj_default"` este de asemenea o opțiune. | Specifică schema de întrețesere care trebuie folosită în Circuit-ul cuantic, urmând convențiile comune Qiskit și [convențiile ffsim pentru ansatz-ul LUCJ](https://qiskit-community.github.io/ffsim/how-to-guides/simulate-lucj.html).                                                                                                                                                                                       |
| `"ansatz_reps"`                   | `int`                               | `2`                              | Mai mare decât 0.                                                                                                                                                | Numărul de repetări ale fiecărui strat din Circuit-ul cuantic.                                                                                                                                                                                                                                                                                                                                                                                                                                                                         |
| `"amplitude_screening_tolerance"` | `Union[float,int]`                  | `0`                              | Cel puțin 0 și mai mic decât 1.                                                                                                                                   | Toleranța pentru deciderea căror stări ar trebui eliminate din subspațiu după diagonalizare. Specifică pragul de includere pentru stările subspațiului pe baza amplitudinilor calculate ale acestora.                                                                                                                                                                                                                                                                                                                                  |
| `"overlap_screening_tolerance"`   | `float`                             | `1e-2`                           | Între `1e-4` și `1e-1`, inclusiv.                                                                                                                          | Toleranța pentru predicția căror stări ar trebui eliminate din subspațiu înainte de diagonalizare. Controlează acuratețea amplitudinilor prezise pentru fiecare stare, o valoare mai mică rezultând în predicții mai precise.                                                                                                                                                                                                                                                                                                            |
## Outputs

Funcția returnează un dicționar cu patru chei și valori. Cheile și valorile sunt documentate în tabelul următor:

| Cheie             | Tipul valorii    | Explicație                                                                                                               |
|:----------------|---------------|---------------------------------------------------------------------------------------------------------------------------|
| `"energy"`      | `float`       | Energia aproximativă a stării fundamentale a moleculei.                                                                      |
| `"states"`      | `List[str]`   | Determinanții selectați care formează subspațiul folosit pentru calculul energiei. Sunt în format alfa-beta alternativ. |
| `"eigenvector"` | `List[float]` | Vectorul propriu corespunzător stării fundamentale a subspațiului compus din `"states"`.                                 |
| `"energy_variance"` | `float` | Varianța energiei stării fundamentale a subspațiului compus din `"states"`, oferind o indicație asupra calității soluției. Această valoare este nenegativă, iar o valoare mai mică înseamnă că starea fundamentală a subspațiului aproximează mai bine un eigenstat al Hamiltonianului sistemului. |
| `"energy_history"` | `List[float]` | Energiile calculate la fiecare iterație în timpul procesului de optimizare hibridă, în ordinea în care au fost calculate. Două energii sunt calculate per iterație ca parte a procesului de optimizare SPSA. |
## Exemplu
Primul exemplu arată cum se calculează energia stării fundamentale pentru o moleculă de NH3 folosind algoritmul HI-VQE.
#### Definirea geometriei moleculare și a opțiunilor
Geometria moleculară a NH3 este furnizată cu coordonate carteziene separate prin ";" pentru fiecare atom.

In [3]:
# Define the molecule geometry
geometry = """
N         -0.85188       -0.02741        0.03141;
H          0.16545        0.00593       -0.01648;
H         -1.16348       -0.39357       -0.86702;
H         -1.16348        0.94228        0.06281;
"""

Opțiuni suplimentare pot fi definite și furnizate pentru sistemul molecular în formatul de dicționar următor.

In [4]:
# Configure some options for the job.
molecule_options = {"basis": "sto3g"}
hivqe_options = {"shots": 100, "max_iter": 20}

Execută funcția cu intrările de geometrie și opțiuni.

In [5]:
# Run HI-VQE
job = function.run(
    geometry=geometry,
    # `backend_name` is the name of a backend with at least 16 qubits, for example, "ibm_marrakesh".
    backend_name=backend_name,
    max_states=2000,
    max_expansion_states=10,
    molecule_options=molecule_options,
    hivqe_options=hivqe_options,
)

Este o idee bună să afișezi ID-ul jobului funcției, astfel încât să poată fi furnizat în solicitările de suport dacă ceva merge greșit.

In [6]:
print("Job ID:", job.job_id)

Job ID: 128ee399-7cfc-4be2-91e9-c4abd22b97c7


This example then utilizes 16 qubits with 8 orbitals of sto3g basis for an NH3 molecule.

Check your Qiskit Function workload's [status](/docs/guides/functions#check-job-status) or return [results](/docs/guides/functions#retrieve-results) as follows:

In [7]:
print(job.status())

QUEUED


Acest exemplu utilizează apoi 16 qubiți cu 8 orbitali ai bazei sto3g pentru o moleculă de NH3.
Verifică [starea](/guides/functions#check-job-status) sau returnează [rezultatele](/guides/functions#retrieve-results) workload-ului tău Qiskit Function după cum urmează:

In [8]:
result = job.result()

# Output can be long, so we display a shortened representation
shortened_result = reprlib.repr(result)
print(shortened_result)

{'eigenvector': [0.9824200343205695, 0.009520846167419264, 6.365368845740147e-08, 3.6832123006425615e-07, 0.0012869941182066654, 2.3403891050875465e-05, ...], 'energy': -55.521146537970466, 'energy_history': [-55.52091922342852, -55.52113695367561, -55.521146537970466, -55.52114653864798, -55.521146537970466, -55.521146537970466, ...], 'energy_variance': 9.788555455342562e-12, ...}


To access the ground state energy, use the "energy" key. The "eigenvector" key provides the CI coefficients with corresponding bitstring notation of electron configuration stored with "states" of the results.

In [10]:
fci_energy = -55.521148034704126  # the exact energy using FCI method
hivqe_energy = result["energy"]
print(
    f"|Exact Energy - HI-VQE Energy|: {abs(fci_energy - hivqe_energy) * 1000} mHa"
)
print(f"Sampled Number of States: {len(result['states'])}")

|Exact Energy - HI-VQE Energy|: 0.0014967336596782843 mHa
Sampled Number of States: 1936


După finalizarea jobului, rezultatele pot fi obținute cu instanța `result()`.

In [11]:
# This cell is hidden from users
backend_name = service.least_busy(operational=True, min_num_qubits=38).name

In [12]:
# Define Li2S geometries
Li2S_geoms = {
    "Li2S_1.51": "S        -1.239044    0.671232   -0.030374;Li       -1.506327    0.432403   -1.498949;Li       -0.899996    0.973348    1.826768;",
    "Li2S_2.40": "S        -1.741432    0.680397    0.346702;Li       -0.529307    0.488006   -1.729343;Li       -1.284307    0.989409    2.177209;",
    "Li2S_3.80": "S        -2.707255    0.674298    0.909161;Li        0.079218    0.552012   -1.671656;Li       -0.927010    0.931502    1.557063;",
}

# Configure some options for the job.
molecule_options = {
    "basis": "sto3g",
}
hivqe_options = {
    "shots": 100,
    "max_iter": 20,
}

results = []
for geom in ["Li2S_1.51", "Li2S_2.40", "Li2S_3.80"]:
    # Run HI-VQE
    job = function.run(
        geometry=Li2S_geoms[geom],
        backend_name=backend_name,  # can use any device with at least 38 qubits
        max_states=2000,
        max_expansion_states=10,
        molecule_options=molecule_options,
        hivqe_options=hivqe_options,
    )
    results.append(job.result())

Pentru a accesa energia stării fundamentale, folosește cheia `"energy"`. Cheia `"eigenvector"` furnizează coeficienții CI cu notația bitstring corespunzătoare a configurației electronice stocate cu `"states"` din rezultate.

In [13]:
job = function.run(
    geometry="invalid-geometry",  # This will cause an error
    backend_name=backend_name,
    max_states=2000,
    max_expansion_states=15,
    molecule_options=molecule_options,
    hivqe_options=hivqe_options,
)

job.result()

QiskitServerlessException: {"message": "An unexpected error occurred during job execution. Please make sure that your inputs are valid. If you are still experiencing problems, you can contact the Qunova Computing support service at qiskit.support@qunovacomputing.com and provide the Function job ID of this job for more assistance.", "status": "failure"}

In [14]:
job.status()

'ERROR'

Output:

|Exact Energy - HI-VQE Energy|: 0.077237504 mHa
Sampled Number of States: 1936
## Performanță
Această secțiune prezintă calculele de referință demonstrate ale HI-VQE pentru un caz cu 24 de qubiți pentru Li2S, un caz cu 40 de qubiți pentru o moleculă N2 și un caz cu 44 de qubiți pentru un sistem FeP-NO.
#### Curba suprafeței de energie potențială de disociere pentru o moleculă Li2S cu 24 de qubiți
Curba PES este prezentată cu referința FCI și estimarea inițială din RHF, împreună cu eroarea de energie față de referința FCI.

![Imagine care arată că HI-VQE produce soluții în limita acurateței chimice față de o curbă PES de referință clasică pentru sistemul Li2S.](../docs/images/guides/qunova-chemistry/Li2S_PES.avif)

Calculele au fost efectuate cu următoarele geometrii și opțiuni.